In [3]:
import os
import glob
import shutil
import random

# =======================
# CONFIG
# =======================

image_dir = "/pscratch/sd/x/xchong/sam3_finetune/seg_annotation_pipeline2/data/images"
mask_dir  = "/pscratch/sd/x/xchong/sam3_finetune/seg_annotation_pipeline2/data/instance_masks_0"

output_root = "./42tiff_flow_threshold0"
train_ratio = 0.8
random_seed = 42

# =======================
# SETUP
# =======================

random.seed(random_seed)

train_dir = os.path.join(output_root, "train")
test_dir  = os.path.join(output_root, "test")

os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# =======================
# FIND IMAGE FILES
# =======================

image_files = glob.glob(os.path.join(image_dir, "*.tif")) + \
              glob.glob(os.path.join(image_dir, "*.tiff"))

image_files = sorted(image_files)

print(f"Found {len(image_files)} images")

# =======================
# MATCH IMAGE + MASK
# =======================

pairs = []

for img_path in image_files:
    base = os.path.splitext(os.path.basename(img_path))[0]
    mask_name = base + "_masks.tif"
    mask_path = os.path.join(mask_dir, mask_name)

    if os.path.exists(mask_path):
        pairs.append((img_path, mask_path))
    else:
        print(f"WARNING: mask not found for {base}")

print(f"Matched {len(pairs)} image-mask pairs")

# =======================
# TRAIN / TEST SPLIT
# =======================

random.shuffle(pairs)

split_idx = int(len(pairs) * train_ratio)

train_pairs = pairs[:split_idx]
test_pairs  = pairs[split_idx:]

print(f"Train: {len(train_pairs)}")
print(f"Test : {len(test_pairs)}")

# =======================
# COPY FILES
# =======================

def copy_pairs(pairs, target_dir):
    for img_path, mask_path in pairs:
        shutil.copy2(img_path, target_dir)
        shutil.copy2(mask_path, target_dir)

copy_pairs(train_pairs, train_dir)
copy_pairs(test_pairs, test_dir)

print("Dataset created successfully!")


Found 42 images
Matched 42 image-mask pairs
Train: 33
Test : 9
Dataset created successfully!
